In [ ]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

# Index

In [ ]:
!pip3 install -q swifter
import pandas as pd
import dask.dataframe
import swifter
from tqdm import tqdm

tqdm.pandas()

## Make Index Smaller

In [ ]:
import os
import json
from samantha.utils.hdfs_helper import hdfs_ls


def remove_dict(r):
    start_idx = r.find('"everynoise_genre"')
    r = r[start_idx:]
    r = "{" + r
    return r


out_fp = "index_2"
os.makedirs(out_fp, exist_ok=True)

fps = hdfs_ls("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_everynoise_N937k_Lmix_mp3/index_1")

for fp in tqdm(fps):
    shard_fp = os.path.join(out_fp, os.path.basename(fp))
    if os.path.exists(shard_fp):
        continue
    
    df = pd.read_parquet(fp)
    df["meta"] = df["meta"].swifter.apply(remove_dict)
#     df["everynoise_genre"] = df.meta.apply(lambda i: i.get("everynoise_genre"))
#     df["everynoise_trending"] = df.meta.apply(lambda i: i.get("everynoise_trending"))
    
    
    df.to_parquet(
        shard_fp,
        index=None,
    )

In [ ]:
import dask.dataframe
from tqdm import tqdm


# ddf = dask.dataframe.read_parquet("index__tmp")
ddf = dask.dataframe.read_parquet("index_2")
results = ddf.compute()

In [ ]:
results

In [ ]:
results.meta = results.meta.apply(json.loads)

In [ ]:
results.meta.values[0]

In [ ]:
results["everynoise_genre_merged"] = results.meta.apply(lambda i: i.get("everynoise_genre_merged"))
assert results["everynoise_genre_merged"].isna().sum() == 0

In [ ]:
results["everynoise_genre"] = results.meta.apply(lambda i: i.get("everynoise_genre"))
results["everynoise_trending"] = results.meta.apply(lambda i: i.get("everynoise_trending"))
results["everynoise_trending_genre"] = results["everynoise_trending"].apply(lambda r: r.get("genre") if r is not None else None)  # replace vantage/genre dict with genre

print(results[["everynoise_genre", "everynoise_trending_genre"]].isna().all(axis=1).sum())

In [ ]:
results["everynoise_genre_merged"] = results["everynoise_genre"].fillna(results["everynoise_trending_genre"])
print(results["everynoise_genre_merged"].isna().sum())

## Hard Filters

In [ ]:
results = results.dropna(subset=["everynoise_genre_merged"])

## Write Index

In [ ]:
index_df = results.reset_index(drop=True)

In [ ]:
index_df["meta"] = index_df.apply(
    lambda row: {
        "everynoise_genre": row["everynoise_genre"],
        "everynoise_trending_genre": row["everynoise_trending_genre"],
        "everynoise_genre_merged": row["everynoise_genre_merged"],
    }, 
    axis=1
)


In [ ]:
index_df = index_df[["uttid", "meta", "data_file", "text", "row_group_no"]]
index_df["meta"] = index_df["meta"].progress_apply(json.dumps)

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm


groups = index_df.groupby("data_file")
index_version = "index_2"
for name, df in tqdm(groups, total=len(groups)):
    
    if not len(df):
        print(f"Skipped {name}")
        continue
        
    name = Path(name)
    
    index_name = f"{name.stem}"
    index_fp = f"{os.path.join(index_version, index_name)}.parquet"
    os.makedirs(os.path.dirname(index_fp), exist_ok=True)
    df.to_parquet(
        index_fp,
        index=None,
    )

In [ ]:
from recipes.research.dataset.collection import EveryNoiseParquetDataset, EveryNoiseGenreParquetDataset
from samantha.data.audio.dataset import AudioFolderDataModule

batch_size = 8
num_workers = 16
sample_rate = 44100
segment_duration = 30

everynoise = EveryNoiseParquetDataset(
    sample_rate=sample_rate,
    channels=2,
    segment_duration=segment_duration,
    resampled=True,
    shardshuffle=True,
)
datamodule = AudioFolderDataModule([everynoise], [], [], weights=None, batch_size=batch_size, shuffle=None, num_workers=num_workers)
train_loader = datamodule.train_dataloader()

In [ ]:
batch = next(iter(train_loader))

In [ ]:
from IPython.display import display, Audio

batch_idx = 4
n_frames = batch.segment_info[batch_idx].n_frames
display(Audio(batch.audio[batch_idx, :, :n_frames], rate=sample_rate))

In [ ]:
from samantha.transforms.audio import batch_plot_spectrogram, MelSpectrogram


mel_transform = MelSpectrogram(
    sample_rate=sample_rate,
    n_mels=160,
    n_fft=4096,
).to(batch.audio.device)

mel, _ = mel_transform(batch.audio.mean(dim=1))
batch_plot_spectrogram(mel.cpu(), plot_log=True, figsize=(20, 20))